# Financial Health Dashboard: Tesla vs BYD vs Ford
Analyzing EV transition impact across three automotive archetypes
- Data Source: Yahoo Finance via yfinance
- Period: 2019-2024

## Data Limitations

1. **Historical depth**: yfinance free tier returns a maximum of 4 years of annual 
fundamentals. After removing the partial current year and NaN rows, this analysis 
covers **2022–2024 (3 years)**. This is sufficient to capture key trends in the 
EV transition period but limits long-term pattern analysis.

2. **Fiscal year assumption**: All three companies (Tesla, BYD, Ford) report on a 
calendar year basis (January–December). The partial year filter uses current 
calendar year as the cutoff. This logic will need adjustment for companies with 
non-December fiscal year ends.

**Future improvement**: Supplement with SimFin or SEC filings to extend history 
to 2019 for a fuller pre/post-EV boom comparison.

In [1]:
# Install if needed
# !pip install yfinance pandas

import yfinance as yf
import pandas as pd

# Define companies
tickers = {
    "Tesla": "TSLA",
    "BYD": "BYDDY",  # BYD's US-listed ADR
    "Ford": "F"
}

In [2]:
# Pull financial statements for all three companies
data = {}

for name, ticker in tickers.items():
    company = yf.Ticker(ticker)
    data[name] = {
        "income": company.income_stmt,
        "balance_sheet": company.balance_sheet,
        "cashflow": company.cash_flow
    }
    print(f"{name} data pulled successfully")

Tesla data pulled successfully


BYD data pulled successfully


Ford data pulled successfully


In [3]:
# Inspect Tesla's income statement first
print("=== TESLA INCOME STATEMENT ===")
print(data["Tesla"]["income"])
print("\n=== COLUMNS ===")
print(data["Tesla"]["income"].index.tolist())

=== TESLA INCOME STATEMENT ===
                                                      2025-12-31  \
Tax Effect Of Unusual Items                        -1.333800e+08   
Tax Rate For Calcs                                  2.700000e-01   
Normalized EBITDA                                   1.225800e+10   
Total Unusual Items                                -4.940000e+08   
Total Unusual Items Excluding Goodwill             -4.940000e+08   
Net Income From Continuing Operation Net Minori...  3.794000e+09   
Reconciled Depreciation                             6.148000e+09   
Reconciled Cost Of Revenue                          7.773300e+10   
EBITDA                                              1.176400e+10   
EBIT                                                5.616000e+09   
Net Interest Income                                 1.342000e+09   
Interest Expense                                    3.380000e+08   
Interest Income                                     1.680000e+09   
Normalized Income

In [4]:
# Inspect Tesla's cash flow statement and balance sheet columns
print("=== TESLA CASHFLOW COLUMNS ===")
print(data["Tesla"]["cashflow"].index.tolist())

print("\n=== TESLA BALANCE SHEET COLUMNS ===")
print(data["Tesla"]["balance_sheet"].index.tolist())

=== TESLA CASHFLOW COLUMNS ===
['Free Cash Flow', 'Repayment Of Debt', 'Issuance Of Debt', 'Issuance Of Capital Stock', 'Capital Expenditure', 'Interest Paid Supplemental Data', 'Income Tax Paid Supplemental Data', 'End Cash Position', 'Beginning Cash Position', 'Effect Of Exchange Rate Changes', 'Changes In Cash', 'Financing Cash Flow', 'Cash Flow From Continuing Financing Activities', 'Net Other Financing Charges', 'Proceeds From Stock Option Exercised', 'Net Common Stock Issuance', 'Common Stock Issuance', 'Net Issuance Payments Of Debt', 'Net Long Term Debt Issuance', 'Long Term Debt Payments', 'Long Term Debt Issuance', 'Investing Cash Flow', 'Cash Flow From Continuing Investing Activities', 'Net Other Investing Changes', 'Net Investment Purchase And Sale', 'Sale Of Investment', 'Purchase Of Investment', 'Net Business Purchase And Sale', 'Sale Of Business', 'Purchase Of Business', 'Net Intangibles Purchase And Sale', 'Sale Of Intangibles', 'Purchase Of Intangibles', 'Net PPE Purch

In [5]:
import sys
sys.path.append('..')  # Allows importing from project root
from metrics import net_profit_margin, return_on_assets

metrics = {}

for name in tickers.keys():
    income = data[name]["income"]
    balance = data[name]["balance_sheet"]
    cashflow = data[name]["cashflow"]
    
    npm = net_profit_margin(income.loc["Net Income"], income.loc["Total Revenue"])
    roa = return_on_assets(income.loc["Net Income"], balance.loc["Total Assets"])
    fcf = cashflow.loc["Free Cash Flow"]
    
    metrics[name] = pd.DataFrame({
        "Net Profit Margin (%)": npm,
        "Free Cash Flow": fcf,
        "Return on Assets (%)": roa
    }).sort_index()

print("Metrics calculated using metrics.py")
for name in metrics:
    print(f"\n{name}:")
    print(metrics[name])

Metrics calculated using metrics.py

Tesla:
            Net Profit Margin (%)  Free Cash Flow  Return on Assets (%)
2021-12-31                    NaN             NaN                   NaN
2022-12-31              15.446466    7.552000e+09             15.282130
2023-12-31              15.499158    4.357000e+09             14.067981
2024-12-31               7.298598    3.581000e+09              5.840911
2025-12-31               4.000970    6.220000e+09              2.753146

BYD:
            Net Profit Margin (%)  Free Cash Flow  Return on Assets (%)
2021-12-31                    NaN             NaN                   NaN
2022-12-31               3.919828    4.338080e+10              3.365817
2023-12-31               4.987555    4.763152e+10              4.420707
2024-12-31               5.180056    3.609410e+10              5.138705
2025-12-31               4.057269   -9.767231e+10              3.691062

Ford:
            Net Profit Margin (%)  Free Cash Flow  Return on Assets (%)
2021-12

In [6]:
from datetime import datetime

# Drop NaN rows
for name in metrics:
    metrics[name] = metrics[name].dropna()

# Drop partial years dynamically
# A year is partial if it matches the current year
current_year = datetime.today().year - 2 # Use -2 to be safe, as some companies report with a delay so yfinance uses estimates

for name in metrics:
    metrics[name] = metrics[name][metrics[name].index.year <= current_year]

print("Cleaned date ranges:")
for name in metrics:
    print(f"{name}: {metrics[name].index.tolist()}")

Cleaned date ranges:
Tesla: [Timestamp('2022-12-31 00:00:00'), Timestamp('2023-12-31 00:00:00'), Timestamp('2024-12-31 00:00:00')]
BYD: [Timestamp('2022-12-31 00:00:00'), Timestamp('2023-12-31 00:00:00'), Timestamp('2024-12-31 00:00:00')]
Ford: [Timestamp('2022-12-31 00:00:00'), Timestamp('2023-12-31 00:00:00'), Timestamp('2024-12-31 00:00:00')]


In [7]:
import pickle

# Save metrics to data folder
with open('../data/metrics.pkl', 'wb') as f:
    pickle.dump(metrics, f)
    
print("Metrics saved to data/metrics.pkl")

Metrics saved to data/metrics.pkl


In [8]:
# Save raw data for use in later notebooks
with open('../data/raw_data.pkl', 'wb') as f:
    pickle.dump(data, f)
    
print("Raw data saved to data/raw_data.pkl")

Raw data saved to data/raw_data.pkl


## Phase 2: Quarterly Fundamentals via SEC EDGAR

The yfinance pull above gives only ~3 years of *annual* data. To analyze the EV
transition quarter-by-quarter we pull XBRL facts directly from SEC EDGAR
(`data.sec.gov`, no API key required). All extraction logic lives in
`edgar_utils.py` (tested in `test_edgar_utils.py`); this notebook just orchestrates.

**BYD is excluded** — as a foreign private issuer it files a 20-F annually and has
no 10-Q quarterly reports, so quarterly EDGAR data doesn't exist for it.

Each metric is extracted according to how the SEC requires it to be reported:

| Metric | Reporting type | Why |
| --- | --- | --- |
| Net Income | discrete quarter | income statement reports each standalone 3-month quarter; Q4 = FY − Q1−Q2−Q3 only when not reported directly |
| Revenue | discrete quarter, merged tags | spans the `Revenues` → ASC 606 `RevenueFromContractWithCustomerExcludingAssessedTax` tag migration |
| Operating Cash Flow | year-to-date | cash-flow statement is cumulative; discrete quarters recovered by differencing (Q2 = 6mo−Q1, etc.) |
| Capex | year-to-date | same cash-flow convention; Ford uses `PaymentsToAcquireProductiveAssets` vs Tesla's `...PropertyPlantAndEquipment` |
| Assets | instant | balance-sheet item is a point-in-time snapshot (no duration); no reconstruction needed |

Output is a tidy long DataFrame `[company, metric, end, val]` saved to
`data/quarterly_edgar.pkl`.

### The core problem: a "quarter" isn't reported the same way twice

Pulling clean quarterly fundamentals from XBRL is harder than it looks because the SEC
never hands you four tidy quarters. Two accounting properties decide how each number has
to be handled, and the table above is just these two axes applied to five metrics:

- **Flow vs. stock.** *Flows* (income, cash flow) accumulate over a span of time; *stocks*
  (balance-sheet items like Assets) exist at a single instant. A flow has a `start` and an
  `end`; a stock has only an `end`. You can sum or difference flows; you can't do either to a
  stock — the year-end Assets figure is simply read off, never reconstructed.
- **Discrete vs. cumulative.** Among flows, the income statement reports each quarter
  *standalone* (a clean 3-month figure), but the cash-flow statement reports *year-to-date*
  (Q1 = 3mo, the Q2 filing = 6mo, Q3 = 9mo, 10-K = 12mo). Discrete quarters therefore need
  **subtraction** to recover (`Q4 = FY − Q1−Q2−Q3` for the income statement; differencing
  consecutive YTD periods for cash flow).

Get either axis wrong and the numbers are silently off — e.g. treating cumulative cash flow
as if it were discrete would roughly *triple* Q3. Reconstructed values are then verified, not
trusted: the validation cell below reconciles them against the figures each company actually
filed.

In [9]:
import edgar_utils as eu  # uses sys.path.append('..') from the metrics cell above

# CIKs for the two domestic filers. BYD is excluded by design: as a foreign private
# issuer it files a 20-F annually and never files 10-Qs, so no quarterly XBRL exists.
ciks = {"Tesla": "1318605", "Ford": "37996"}


# Design decision: a "quarter" is NOT reported uniformly across the three financial
# statements, so each metric is dispatched to the extractor that matches its reporting
# shape rather than forced through one generic parser. The (method, concept) pair makes
# that decision explicit and auditable:
#   full    -> income-statement flow: discrete 3-month quarters; Q4 reconstructed from
#              the 10-K (FY - Q1 - Q2 - Q3) only in years it isn't reported directly
#   merged  -> same shape, but stitched across a tag migration (the ASC 606 revenue rename)
#   ytd     -> cash-flow flow: reported cumulatively year-to-date, differenced into quarters
#   instant -> balance-sheet stock: a point-in-time snapshot, nothing to reconstruct
def extract_quarterly(name, facts):
    """Extract all five quarterly metrics for one company into tidy long rows."""
    # Tesla and Ford book the same economic outflow (capex) under different XBRL tags,
    # so the concept is selected per company rather than hard-coded for both.
    capex_tag = ("PaymentsToAcquirePropertyPlantAndEquipment" if name == "Tesla"
                 else "PaymentsToAcquireProductiveAssets")
    specs = {
        "NetIncome":         ("full",    "NetIncomeLoss"),
        "Revenue":           ("merged",  ["Revenues",
                                          "RevenueFromContractWithCustomerExcludingAssessedTax"]),
        "OperatingCashFlow": ("ytd",     "NetCashProvidedByUsedInOperatingActivities"),
        "Capex":             ("ytd",     capex_tag),
        "Assets":            ("instant", "Assets"),
    }
    frames = []
    for metric, (method, concept) in specs.items():
        if method == "full":
            q, _ = eu.get_full_metric_history(facts, concept)
        elif method == "merged":
            q, _ = eu.get_merged_metric_history(facts, concept)
        elif method == "ytd":
            q, _ = eu.get_ytd_flow_history(facts, concept)
        elif method == "instant":
            q = eu.get_instant_metric_history(facts, concept)
        # Annual series (the discarded `_`) is intentionally not saved here — it is
        # re-derived in the validation cell below to reconcile against these quarters.
        frames.append(q.assign(company=name, metric=metric)[["company", "metric", "end", "val"]])
    return pd.concat(frames, ignore_index=True)


# Pull each company's full fact set once and keep it, so the validation cell can
# reconcile against reported annuals without re-downloading the (large) companyfacts JSON.
facts_by_company = {name: eu.get_company_facts(cik) for name, cik in ciks.items()}

parts = [extract_quarterly(name, facts) for name, facts in facts_by_company.items()]
quarterly_edgar = (pd.concat(parts, ignore_index=True)
                   .sort_values(["company", "metric", "end"])
                   .reset_index(drop=True))

print("Quarters extracted per company/metric:")
print(quarterly_edgar.groupby(["company", "metric"]).size())
quarterly_edgar.head()

Quarters extracted per company/metric:
company  metric           
Ford     Assets               69
         Capex                71
         NetIncome            47
         OperatingCashFlow    37
         Revenue              71
Tesla    Assets               61
         Capex                63
         NetIncome            65
         OperatingCashFlow    50
         Revenue              65
dtype: int64


,company,metric,end,val
0,Ford,Assets,2008-12-31,216052000000
1,Ford,Assets,2009-06-30,200190000000
2,Ford,Assets,2009-09-30,203106000000
3,Ford,Assets,2009-12-31,192040000000
4,Ford,Assets,2010-03-31,191968000000


### Validation — don't trust derived data

Q4 and the YTD-differenced quarters are *reconstructed*, not reported, so they have to be
checked against figures each company actually filed before anything downstream relies on them.
The check is a **reconciliation identity**: for any fully-covered year, the four quarters must
sum back to the reported annual. This is also exactly what would catch the earlier
Q4-double-counting bug — a stray second Dec-31 row makes the annual sum overshoot.

In [10]:
# Validation — reconstructed quarters must reconcile to the figures the company filed.
# Net Income is the strict test: it carries the reconstructed Q4 (FY - Q1 - Q2 - Q3),
# so if reconstruction is wrong the four quarters won't sum back to the reported annual.
ANALYSIS_START = 2018   # the EV-transition window this project actually analyses
TOL = 0.001             # 0.1% — absorbs rounding in as-filed XBRL


def reconcile(facts, concept):
    """Per-year: sum of the four quarters vs the reported annual, as a relative error."""
    q, a = eu.get_full_metric_history(facts, concept)
    by_year = q.assign(yr=q['end'].dt.year).groupby('yr')['val'].agg(qsum='sum', n='count')
    annual = a.assign(yr=a['end'].dt.year).set_index('yr')['val'].rename('reported_annual')
    chk = by_year.join(annual)
    chk = chk[chk['n'] == 4].dropna()                       # only fully-covered years
    chk['rel_err'] = (chk['qsum'] - chk['reported_annual']).abs() / chk['reported_annual'].abs()
    return chk


for name, facts in facts_by_company.items():
    chk = reconcile(facts, 'NetIncomeLoss')
    window = chk[chk.index >= ANALYSIS_START]
    print(f"{name} NetIncome — analysis window {ANALYSIS_START}+: "
          f"{len(window)} years, worst rel. error {window['rel_err'].max():.4%}")
    # Hard requirement: every year that feeds the analysis must reconcile.
    assert (window['rel_err'] < TOL).all(), f"{name}: a {ANALYSIS_START}+ year fails to reconcile"
    # Soft check: surface (don't fail on) older anomalies — these trace to a later 10-K
    # re-tagging a prior year; we keep the latest-filed annual to honour restatements.
    stale = chk[(chk.index < ANALYSIS_START) & (chk['rel_err'] >= TOL)]
    if not stale.empty:
        print(f"  pre-{ANALYSIS_START} reconciliation anomalies (informational): "
              f"{[int(y) for y in stale.index]}")

# Integrity: the saved frame must not contain any period-end twice — the exact failure
# mode of the earlier Q4 double-counting bug.
dupes = quarterly_edgar.duplicated(subset=['company', 'metric', 'end']).sum()
print(f"\nDuplicate (company, metric, end) rows in saved frame: {dupes}")
assert dupes == 0
print("Validation passed: analysis-window quarters reconcile and no duplicate periods.")

Tesla NetIncome — analysis window 2018+: 8 years, worst rel. error 0.0000%
Ford NetIncome — analysis window 2018+: 3 years, worst rel. error 0.0000%
  pre-2018 reconciliation anomalies (informational): [2012, 2013]

Duplicate (company, metric, end) rows in saved frame: 0
Validation passed: analysis-window quarters reconcile and no duplicate periods.


**Reading the result.** Within the 2018+ EV-transition window — the only data this project
actually analyses — every reconstructed quarter sums back to the reported 10-K annual *to the
dollar* (0.00% error) for both companies, and the saved frame has no duplicated period-ends.

The two pre-2018 Ford flags are a deliberately-surfaced data-quality finding, not a bug:
Ford's FY2013 net income is tagged as **$7.16B** in its original 2013 10-K but as **$11.95B**
in a comparative column of the 2015 10-K filed three years later. `get_full_metric_history`
keeps the *latest-filed* value (the right rule for genuine restatements), so it picks up that
later re-tagging — which happens to disagree with both the original filing and Ford's own
quarters. Because it predates the analysis window and the affected year reports Q4 directly
(no reconstruction depends on it), it's logged for transparency rather than silently dropped.

**Decisions & data-quality log**

| Decision | Rationale |
| --- | --- |
| SEC EDGAR over yfinance for quarterly | yfinance gives ~3 yrs of *annual* data; XBRL gives 15+ yrs of quarters, free, no key |
| One extractor per reporting shape | flow/stock and discrete/cumulative are different problems; a single parser would silently mis-handle one |
| Reconstruct Q4 only when not reported | some 10-Ks file a discrete Q4; reconstructing anyway emitted two Dec-31 rows (the double-count bug, now guarded + dup-checked) |
| Keep latest-filed value on duplicate `end` | honours restatements — at the documented cost of the Ford 2013 re-tag above |
| Assert hard on 2018+, surface older anomalies | validates the data that feeds the analysis without failing on irrelevant historical XBRL noise |
| Exclude BYD from quarterly | foreign private issuer files 20-F only — no 10-Qs exist, so quarterly data is impossible, not merely missing |

In [11]:
# Save quarterly EDGAR data for use in later notebooks
quarterly_edgar.to_pickle('../data/quarterly_edgar.pkl')
print(f"Saved {len(quarterly_edgar)} rows to data/quarterly_edgar.pkl")

Saved 599 rows to data/quarterly_edgar.pkl
